# 📷 Google Photos Takeout 雲端自動化增量整理工坊 (高視覺即時進度版)

本筆記本專為 **Google One 2TB** 雲端生態設計，直接結合 `14_Google-Photos-Takeout-Organizer` 官方核心引擎。

### 核心亮點：
1. **即時百分比進度顯示**：解壓、配對、歸檔每一步都有清晰進度條與檔案計數，不再盲目等待！
2. **全自動掛載 Google Drive**：直接在 Google 雲端內部高速處理，本機電腦 0 下載、0 流量。
3. **全域 Sidecar 配對 & 五層日期保底**：解析 JSON 時間戳記，精準校正相片建立時間。
4. **iPhone Live Photo 綁定**：`.HEIC` 與 `.MOV` 同步歸檔，動態照片不拆散。
5. **SHA-256 增量去重**：重複照片自動略過，不浪費 2TB 雲端空間。
6. **自動安全清理**：驗證 100% 寫入無誤後，自動清理 Takeout 暫存 ZIP 與工作目錄。

## 步驟 1：掛載 Google 雲端硬碟
點擊左側播放鍵 ▶️，依提示授權存取 Google Drive：

In [ ]:
from google.colab import drive
import os, sys, shutil, glob
from pathlib import Path

drive.mount('/content/drive')
print('✓ Google Drive 掛載成功！')

## 步驟 2：安裝 14 號專案核心引擎與相依套件

In [ ]:
!pip install -q pillow pillow-heif
!pip install -q git+https://github.com/lianghao02/14_Google-Photos-Takeout-Organizer.git
print('✓ 核心整理引擎安裝完成！')

## 步驟 3：設定路徑與自動掃描 Takeout 壓縮檔

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
TAKEOUT_DIR = DRIVE_ROOT / 'Takeout'
WORK_DIR = DRIVE_ROOT / 'Takeout_Work'
OUTPUT_DIR = DRIVE_ROOT / 'Organized_Photos'

TAKEOUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

zip_files = sorted(list(TAKEOUT_DIR.glob('*.zip')))

total_size_gb = sum(z.stat().st_size for z in zip_files) / (1024 ** 3)
print(f'相簿目標目錄：{OUTPUT_DIR}')
print(f'發現待處理 Takeout 壓縮檔數量：{len(zip_files)} 卷 (總計約 {total_size_gb:.1f} GB)')
for idx, z in enumerate(zip_files[:5], 1):
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f'  [{idx:03d}] {z.name} ({size_mb:.1f} MB)')
if len(zip_files) > 5:
    print(f'  ... 略過中間 {len(zip_files)-5} 個檔案 ...')
    last_z = zip_files[-1]
    print(f'  [{len(zip_files):03d}] {last_z.name} ({last_z.stat().st_size / (1024*1024):.1f} MB)')

if not zip_files:
    print('\n⚠️ 目前 Takeout 目錄下沒有 ZIP 檔案。若 Google 尚未完成打包，請稍候再試。')

## 步驟 4：執行分析、規劃、去重與無損匯出 (含即時進度顯示)

In [ ]:
if not zip_files:
    raise RuntimeError('沒有待處理的 ZIP 檔案，流程終止。')

import time
from google_photos_takeout_organizer.service import analyze
from google_photos_takeout_organizer.exporter import export
from google_photos_takeout_organizer.verifier import verify

# 定義即時進度回呼函式
last_update = [0]
def console_progress(stage, current, total, message):
    now = time.time()
    pct = (current / total * 100) if total else 0
    # 限制更新頻率為每 0.3 秒一次，避免洗螢幕
    if now - last_update[0] > 0.3 or current == total or current == 1:
        last_update[0] = now
        prefix = '📦 解壓封存檔' if stage == 'ANALYZE_EXTRACT' else ('🔎 分析配對中' if stage == 'ANALYZE' else '📁 歸檔寫入中')
        # 使用 \r 實現原地動態更新進度條
        sys.stdout.write(f'\r{prefix} [{pct:5.1f}%] ({current}/{total}) - {message[:50]}')
        sys.stdout.flush()
        if current == total:
            sys.stdout.write('\n')

print('====================================================')
print('1/3 階段：解壓 ZIP、全域 Sidecar 配對與五層日期校準...')
print('====================================================')
manifest = analyze(zip_files, WORK_DIR, progress_callback=console_progress)
manifest_path = WORK_DIR / 'manifest.json'
print('\n✓ 1/3 階段分析完成！摘要：')
for k, v in manifest['summary'].items():
    print(f'  - {k}: {v}')

print('\n====================================================')
print('2/3 階段：執行無損增量歸檔 (去重複與年月分類)...')
print('====================================================')
export_result = export(manifest_path, OUTPUT_DIR, progress_callback=console_progress)
print('\n✓ 2/3 階段歸檔完成！摘要：')
for k, v in export_result['summary'].items():
    print(f'  - {k}: {v}')

print('\n====================================================')
print('3/3 階段：校驗匯出成果 (SHA-256 與位元組完整性)...')
print('====================================================')
verify_result = verify(manifest_path, OUTPUT_DIR)
print('\n驗證結果：')
print(verify_result)

if verify_result.get('status') == 'OK':
    print('\n🎉 所有相片與影片 100% 完整無誤寫入！')
else:
    print('\n⚠️ 部分檔案校驗未通過，請查看上方訊息。')

## 步驟 5：安全清理 Takeout 壓縮檔與暫存區 (釋放雲端硬碟空間)
**注意**：僅在校驗 100% 通過時才會刪除原始 ZIP。

In [ ]:
# 設定是否刪除原始 ZIP 檔案 (預設為 True 釋放空間)
DELETE_SOURCE_ZIPS = True

if verify_result.get('status') == 'OK':
    if DELETE_SOURCE_ZIPS:
        for z in zip_files:
            try:
                z.unlink()
                print(f'✓ 已安全刪除來源封存檔：{z.name}')
            except Exception as e:
                print(f'刪除失敗 {z.name}: {e}')
    
    # 清理工作目錄中的暫存 extracted 檔案，保留 manifest 與 report
    extracted_dir = WORK_DIR / 'extracted'
    if extracted_dir.exists():
        shutil.rmtree(extracted_dir, ignore_errors=True)
        print('✓ 已清理 Takeout_Work/extracted 暫存目錄。')
        
    print('\n✨ 任務圓滿完成！雲端硬碟空間已釋放，相片已妥善歸檔至 Organized_Photos/')
else:
    print('❌ 因驗證未完全通過，已嚴格保留所有原始 ZIP 檔案與暫存區，確保資料絕對安全！')